# Customer Churn Prediction

## 1. Project Overview

### 🧠 Problem Statement

Customer churn occurs when customers stop using a company’s service. For subscription-based businesses, predicting churn is critical for customer retention and long-term profitability.

**Goal:** Build a machine learning model that predicts whether a customer will churn, using demographic and service-related data.

**Dataset:** Telco Customer Churn Dataset  
Source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

# Load dataset
# raw_df to preserve the original data
raw_df = pd.read_csv('Telco-Customer-Churn.csv')  # make sure it's in the same folder

# Create a copy of the raw data on which we will perform operations
df = raw_df.copy()

# Display first few rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Import Libraries & Load Data

In [ ]:
# Dataset overview
df.info()

# Summary stats
df.describe()

## 3. Data Cleaning & Preparation

### What we need to do:
Before modeling, we must clean the data. This includes fixing data types, handling missing values, and encoding categorical variables.

### What we can do:
- Convert data types (e.g., numeric columns stored as strings)
- Handle missing or invalid values
- Drop uninformative columns
- Encode categorical variables for ML models

### What we use and why:
- `pd.to_numeric()` to fix numeric conversion issues
- `dropna()` to remove invalid rows
- `map()` to convert churn labels to binary
- `pd.get_dummies()` to one-hot encode categoricals (better for tree-based models)


➤ Check for null values:

There are no null values in any of the columns

In [ ]:
df.isnull().sum()

➤ Fix TotalCharges column (some entries are empty strings):

In [ ]:
# Fixing 'TotalCharges': convert to numeric (errors='coerce' turns bad data into NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Now check again
df.isnull().sum()

# Drop rows with missing TotalCharges (they're very few)
df.dropna(subset=['TotalCharges'], inplace=True)

➤ Drop uninformative customerID column:

customerID is just an identifier, so it can be dropped. 

In [ ]:
df.drop(columns=['customerID'], inplace=True)

➤ Encode Churn as binary target:

The target column is Churn, which is Yes/No. We'll convert this to binary. 

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

➤ One-hot encode categorical features:

In [ ]:
# Get dummies for categorical variables
df = pd.get_dummies(df, drop_first=True)

## 4. Sanity Check

### What we need to do:
Confirm that the dataset is clean, numeric, and model-ready.

### What we do:
- Confirm all columns are numeric
- Check for remaining null values
- Ensure target column (`Churn`) is binary

Our dataset is now ready for exploratory data analysis (EDA).

In [ ]:
# Show all object-type (non-numeric) columns
df.select_dtypes(include='object').nunique()

In [ ]:
# Check structure
df.info()

# Confirm no missing values
df.isnull().sum()

# Check target distribution
df['Churn'].value_counts(normalize=True)

## 5. Exploratory Data Analysis (EDA)

### What we need to do:
Explore the data visually and statistically to identify patterns, relationships, and trends related to customer churn.

### What we can do:
- Analyze the distribution of churned vs. retained customers
- Explore relationships between churn and features like tenure, contract type, payment method, etc.
- Identify which features are most informative

### What we use and why:
- `seaborn` and `matplotlib` for clear visualizations
- Count plots and histograms for category analysis
- Box plots and correlation heatmaps for numeric relationships

Our goal is to generate hypotheses and insights that will inform our feature selection and modeling.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent plot style
sns.set(style='whitegrid')

# Churn distribution
plt.figure(figsize=(5,4))
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.xticks([0, 1], ['No', 'Yes'])
plt.show()

### Insight: Month-to-Month Contracts and Churn

Customers on **month-to-month contracts** are significantly more likely to churn compared to those on one- or two-year contracts.

### Why this matters:
- Month-to-month plans offer customers more flexibility, but also less commitment.
- The lack of long-term obligation makes it easier for customers to switch to competitors or cancel the service.

### Business implication:
Retention strategies should prioritize customers on month-to-month contracts. Offering

In [ ]:
# Create a derived column for 'Month-to-month' if it was dropped
df['Contract_Month-to-month'] = ((df['Contract_One year'] == 0) & (df['Contract_Two year'] == 0)).astype(int)

plt.figure(figsize=(6,4))
sns.countplot(x='Contract_Month-to-month', hue='Churn', data=df)
plt.title('Churn by Month-to-Month Contract')
plt.xlabel('Month-to-Month (1 = Yes)')
plt.ylabel('Count')
plt.show()

## Insight: Tenure and Churn

Customers with **shorter tenure** are much more likely to churn than long-term customers.

### Why this matters:
- New customers may still be evaluating the service and are more likely to leave early.
- Long-tenured customers often represent higher satisfaction and stickiness.

### Business implication:
Early engagement and onboarding are critical. Implementing personalized follow-ups, onboarding assistance, and early incentives may reduce churn in the first few months.

In [ ]:
# Churn vs Tenure
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True, element='step')
plt.title('Tenure Distribution by Churn')
plt.show()

## Insight: Fiber Optic Internet and Churn

Customers using **fiber optic internet** have a higher churn rate compared to those using DSL or no internet service.

### Why this matters:
- Fiber customers might have higher expectations for speed, performance, or support.
- They may also face more competitive alternatives.

### Business implication:
It's important to monitor service quality and support responsiveness for fiber users. Offering service guarantees or performance checks may help retain this group.

In [ ]:
# Plot: Churn distribution by Internet Service type
plt.figure(figsize=(7,4))
sns.countplot(x='InternetService_Fiber optic', hue='Churn', data=df)
plt.title('Churn by Internet Service Type')
plt.xlabel('Internet Service Type')
plt.ylabel('Count')
plt.show()

In [ ]:
# Create a new column: Is the customer on fiber optic?
df['FiberOpticUser'] = (df['InternetService_Fiber optic'] == 'Fiber optic').astype(int)

plt.figure(figsize=(5,4))
sns.countplot(x='FiberOpticUser', hue='Churn', data=df)
plt.title('Churn by Fiber Optic Usage')
plt.xlabel('Fiber Optic User (1 = Yes)')
plt.ylabel('Count')
plt.xticks([0, 1], ['No', 'Yes'])
plt.show()

## Insight: Paperless Billing and Churn

Customers who use **paperless billing** are slightly more likely to churn.

### Why this matters:
- These users may be less engaged or receive fewer tangible reminders of service.
- Could correlate with younger, more mobile customers.

### Business implication:
Use email/SMS billing as an opportunity to re-engage customers and add value (e.g., tips, usage summaries, offers).


In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='PaperlessBilling_Yes', hue='Churn', data=df)
plt.title('Churn by Paperless Billing')
plt.xlabel('Paperless Billing')
plt.ylabel('Count')
plt.show()

## Insight: Payment Method and Churn

Customers paying via **electronic check** have the highest churn rate.

### Why this matters:
- These customers may not be on auto-pay, making it easier to leave.
- It may also reflect a demographic with lower trust or financial commitment.

### Business implication:
Incentivizing auto-pay methods (like credit cards or bank transfers) could reduce churn from this high-risk group.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(x='PaymentMethod', hue='Churn', data=df)
plt.title('Churn by Payment Method')
plt.xticks(rotation=20)
plt.xlabel('Payment Method')
plt.ylabel('Count')
plt.show()

### 🔍 Correlation with Churn

### What we did:
We created a correlation heatmap to understand how strongly each feature is linearly related to customer churn.

### What we use:
We used `df.corr()` to compute Pearson correlation coefficients and `seaborn.heatmap()` to visualize them.

### What we found:
- **Positive correlations (likely to increase churn):**
  - `InternetService_Fiber optic` (0.31)
  - `PaymentMethod_Electronic check` (0.30)
  - `MonthlyCharges` and `PaperlessBilling_Yes` (both ~0.19)
- **Negative correlations (likely to reduce churn):**
  - `tenure` (-0.35): Longer tenure means lower churn.
  - `Contract_Two year` and `Contract_One year` (~-0.3 and -0.18): Long-term contracts reduce churn.
  - `OnlineSecurity_Yes`, `TechSupport_Yes`, `Dependents_Yes` also show protective effects.

### Why this matters:
These correlations help identify strong predictors of churn and guide feature importance analysis during modeling.

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12,8))
corr = df.corr()
sns.heatmap(corr[['Churn']].sort_values(by='Churn', ascending=False), annot=True, cmap='coolwarm')
plt.title('Feature Correlation with Churn')
plt.show()

### EDA Summary

- **Class imbalance**: About 26% of customers in the dataset have churned.
- **Tenure effect**: Customers with lower tenure are much more likely to churn.
- **Contract type**: Month-to-month contracts are strongly associated with churn.
- **Other factors**: Paperless billing and certain payment methods also show correlation with churn.

These insights will guide our feature selection and help tune our machine learning models.
